In [1]:
pip install pygeohash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.2 MB/s eta 0:00:00


In [133]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from rescale_to_bangalore import rescale_to_city_bbox
from sklearn.ensemble import RandomForestRegressor as RFR
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor as lgbmr
from collections import defaultdict
import lightgbm as lgb
from google.colab import files
import pandas as pd
import numpy as np
import pickle

In [179]:
df = pd.read_csv("train.csv")
dft = pd.read_csv("test.csv")
df.isna().sum()

,0
Index,0
geohash,0
day,0
timestamp,0
demand,0
RoadType,600
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,2495


In [180]:
print(df["Weather"].value_counts())
print("\n")
print(df["RoadType"].value_counts())
print("\n")
print("total_timestamps", df["timestamp"].value_counts().size)

Weather
Sunny    27717
Rainy    20824
Foggy    20243
Snowy     7718
Name: count, dtype: int64


RoadType
Residential    69230
Street          3909
Highway         3560
Name: count, dtype: int64


total_timestamps 96


In [181]:
dft.head()

,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy


In [182]:
df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [183]:
print("test_data_size",df.shape)
print("train_data_size",dft.shape)

test_data_size (77299, 11)
train_data_size (41778, 10)


#Temporal and geohash preprocessing

In [184]:
def time_change(df):
  dfcopy = df.copy()
  dfcopy["time_minutes"] = pd.to_datetime(dfcopy["timestamp"], format="%H:%M").dt.hour*60 + pd.to_datetime(dfcopy["timestamp"], format="%H:%M").dt.minute
  dfcopy["hour"] = dfcopy["time_minutes"]/60
  dfcopy["sin_hour"] = np.sin(2 * np.pi * dfcopy["hour"]/24)
  dfcopy["cos_hour"] = np.cos(2 * np.pi * dfcopy["hour"]/24)
  dfcopy["sin_minute"] = np.sin(2 * np.pi * dfcopy["time_minutes"]/1440)
  dfcopy["cos_minute"] = np.cos(2 * np.pi * dfcopy["time_minutes"]/1440)
  dfcopy["is_rush_hour"] = dfcopy["hour"].isin([8, 9, 10]).astype(int)
  dfcopy["night_time"] = dfcopy["hour"].isin([23, 1, 2, 4, ]).astype(int)

  return dfcopy

In [185]:
def labelEncode(df):
  df_copy = df.copy()

  df_copy["Temperature"] = df_copy["Temperature"].fillna(np.mean(df_copy["Temperature"]))
  df_copy["NumberofLanes"] = df_copy["NumberofLanes"].fillna(df_copy["NumberofLanes"].median())

  df_copy = df_copy.drop("timestamp", axis = 1)
  return df_copy

In [186]:
def preprocess(df, le_large_vehicles=None, le_landmarks=None, oh_roadtype=None, oh_weather=None, fit_encoders=True):
    df_copy = df.copy()

    if fit_encoders:
        le_large_vehicles = LabelEncoder()
        le_landmarks = LabelEncoder()
        oh_roadtype = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        oh_weather = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

        df_copy["LargeVehicles"] = le_large_vehicles.fit_transform(df_copy["LargeVehicles"])
        df_copy["Landmarks"] = le_landmarks.fit_transform(df_copy["Landmarks"])

        encoded = oh_roadtype.fit_transform(df_copy[["RoadType"]])
        encoded_w = oh_weather.fit_transform(df_copy[["Weather"]])
    else:
        if not all([le_large_vehicles, le_landmarks, oh_roadtype, oh_weather]):
            raise ValueError("Encoders must be provided when fit_encoders is False.")

        df_copy["LargeVehicles"] = le_large_vehicles.transform(df_copy["LargeVehicles"])
        df_copy["Landmarks"] = le_landmarks.transform(df_copy["Landmarks"])


        encoded = oh_roadtype.transform(df_copy[["RoadType"]])
        encoded_w = oh_weather.transform(df_copy[["Weather"]])

    encoded_arr = pd.DataFrame(encoded, columns=oh_roadtype.get_feature_names_out(["RoadType"]), index=df_copy.index)
    encoded_w_arr = pd.DataFrame(encoded_w, columns=oh_weather.get_feature_names_out(["Weather"]), index=df_copy.index)

    df_copy = df_copy.drop(["Weather", "RoadType"], axis=1)
    df_copy = pd.concat([df_copy, encoded_arr, encoded_w_arr], axis=1)

    return df_copy, le_large_vehicles, le_landmarks, oh_roadtype, oh_weather

In [187]:
def Lag(dfn, df):
  dfncopy = dfn.copy()
  dfncopy = dfn.sort_values(["geohash", "day", "timestamp"])
  dfncopy["demand_lag_15"] = df.groupby("geohash")["demand"].shift(1)
  dfncopy["demand_lag_hour"] = df.groupby("geohash")["demand"].shift(4)
  dfncopy["demand_lag_day"] = df.groupby("geohash")["demand"].shift(96)
  dfncopy["demand_roll_mean_hour"] = df.groupby("geohash")["demand"].shift(1).rolling(4).mean()
  dfncopy["demand_roll_std_hour"] = df.groupby("geohash")["demand"].shift(1).rolling(4).std()

  return dfncopy

In [188]:
dfn = time_change(df)
dfn.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,time_minutes,hour,sin_hour,cos_hour,sin_minute,cos_minute,is_rush_hour,night_time
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,0,0.0,0.0,1.0,0.0,1.0,0,0
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0.0,0.0,1.0,0.0,1.0,0,0
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0.0,0.0,1.0,0.0,1.0,0,0
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,0,0.0,0.0,1.0,0.0,1.0,0,0
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0.0,0.0,1.0,0.0,1.0,0,0


In [189]:
from process_geohash import GeohashProcessor

gp = GeohashProcessor(n_clusters=15, zone_prefix_len=5)
dfn = gp.fit_transform(dfn)

In [190]:
dfn.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,...,cos_hour,sin_minute,cos_minute,is_rush_hour,night_time,lat,lon,geo_zone,dist_to_center,geo_cluster
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,...,1.0,0.0,1.0,0,0,-5.484924,90.664673,qp02z,0.168925,1
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,...,1.0,0.0,1.0,0,0,-5.462952,90.686646,qp02z,0.138313,1
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,...,1.0,0.0,1.0,0,0,-5.462952,90.708618,qp08b,0.127315,1
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,...,1.0,0.0,1.0,0,0,-5.462952,90.862427,qp08g,0.150985,10
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,...,1.0,0.0,1.0,0,0,-5.457458,90.675659,qp02z,0.140444,1


In [218]:
all_geohash = pd.concat([
    df["geohash"],
    dft["geohash"]
]).unique()
le_geohash = LabelEncoder()
le_geohash.fit(all_geohash)

LabelEncoder()

In [219]:
dfn["geohash"] = le_geohash.transform(dfn["geohash"])

ValueError: y contains previously unseen labels: 0

In [220]:
le_geohash.classes_[:10]

array(['qp02yc', 'qp02yf', 'qp02yy', 'qp02yz', 'qp02z1', 'qp02z3',
       'qp02z4', 'qp02z5', 'qp02z6', 'qp02z7'], dtype=object)

In [221]:
dfn.columns

Index(['Index', 'geohash', 'day', 'demand', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan', 'display_lat', 'display_lon'],
      dtype='object')

#Lag and Rolling window

In [222]:
dfn = Lag(dfn, df)

KeyError: 'timestamp'

In [223]:
dfn.head()

,Index,geohash,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,time_minutes,hour,...,RoadType_Residential,RoadType_Street,RoadType_nan,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_nan,display_lat,display_lon
36776,36776,0,48,0.046790,1,1,0,22.809028,630,10.50,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.846786,77.423714
37683,37683,0,48,0.021158,1,1,0,16.543659,645,10.75,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.846786,77.423714
2911,2911,0,48,0.005397,3,0,1,11.299877,60,1.00,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,12.846786,77.423714
8010,8010,0,48,0.012944,2,1,1,32.142120,150,2.50,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.846786,77.423714
8905,8905,0,48,0.025961,1,1,0,15.811063,165,2.75,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.846786,77.423714


In [224]:
dfn = labelEncode(dfn)
dfn, le_large_vehicles, le_landmarks, oh_roadtype, oh_weather = preprocess(dfn)

KeyError: "['timestamp'] not found in axis"

In [225]:
dfn = rescale_to_city_bbox(dfn)

In [226]:
dfn.head()

,Index,geohash,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,time_minutes,hour,...,RoadType_Residential,RoadType_Street,RoadType_nan,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_nan,display_lat,display_lon
36776,36776,0,48,0.046790,1,1,0,22.809028,630,10.50,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.846786,77.423714
37683,37683,0,48,0.021158,1,1,0,16.543659,645,10.75,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.846786,77.423714
2911,2911,0,48,0.005397,3,0,1,11.299877,60,1.00,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,12.846786,77.423714
8010,8010,0,48,0.012944,2,1,1,32.142120,150,2.50,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.846786,77.423714
8905,8905,0,48,0.025961,1,1,0,15.811063,165,2.75,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.846786,77.423714


In [227]:
dfn.columns

Index(['Index', 'geohash', 'day', 'demand', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan', 'display_lat', 'display_lon'],
      dtype='object')

#Training and Modeling

In [203]:
Y = dfn["demand"]
X = dfn.drop(["demand", "display_lat", "display_lon"],  axis = 1)

Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size = 0.2, random_state = 2)

print(Xtrain.shape)
print(Ytrain.shape)
print(Xtest.shape)
print(Ytest.shape)

(61839, 34)
(61839,)
(15460, 34)
(15460,)


In [204]:
type(Ytest)

pandas.core.series.Series

In [205]:
model = lgbmr(
    objective = "regression",
    metric = "rsme",
    boosting_type = "gbdt",
    max_depth = 5,
    min_data_in_leaf = 10,
    n_estimator = 500,
    random_state = 42,
    verbose = -1,
    num_leaves = 16,
    bagging_fraction = 0.4,
    feature_fraction = 0.6
)

model.fit(Xtrain, Ytrain)

LGBMRegressor(bagging_fraction=0.4, feature_fraction=0.6, max_depth=5,
              metric='rsme', min_data_in_leaf=10, n_estimator=500,
              num_leaves=16, objective='regression', random_state=42,
              verbose=-1)

In [206]:
Ypred = model.predict(Xtest)
r2 = r2_score(Ytest, Ypred)
print("r2_score", r2)

r2_score 0.9614152988690531


In [207]:
filename = "traffic_model.pkl"

with open(filename, "wb")as f:
  pickle.dump(model, f)

In [208]:
Xtrain.columns

Index(['Index', 'geohash', 'day', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan'],
      dtype='object')

In [209]:
export_cols = list(Xtrain.columns)

In [210]:
def export_for_tableau(df, model, feature_cols, output_path='tableau_dashboard_data.csv'):
    """
    df           : your full feature dataframe (must include lat, lon, geohash,
                   day, timestamp_minute/hour, RoadType, Weather, etc.)
    model        : your trained LGBMRegressor
    feature_cols : the exact list of columns used to train the model (X columns)
    """
    df = df.copy()

    df['predicted_demand'] = model.predict(df[feature_cols])

    df['hour'] = df['time_minutes'] // 60
    df['minute'] = df['time_minutes'] % 60
    df['time_label'] = df['hour'].astype(str).str.zfill(2) + ':' + df['minute'].astype(str).str.zfill(2)

    df['congestion_level'] = pd.cut(
        df['predicted_demand'],
        bins=[-0.01, 0.1, 0.3, 0.6, 1.01],
        labels=['Low', 'Moderate', 'High', 'Severe']
    )

    export_cols = [
        'geohash', 'display_lat', 'display_lon',
        'day', 'time_minutes', 'hour', 'time_label',
        'predicted_demand', 'congestion_level',
        'RoadType', 'Weather', 'Temperature',
        'NumberofLanes', 'LargeVehicles', 'Landmarks'
    ]
    export_cols = [c for c in export_cols if c in df.columns]

    df[export_cols].to_csv(output_path, index=False)
    print(f"Saved {len(df)} rows to {output_path}")
    print(f"Columns: {export_cols}")
    return df[export_cols]

In [211]:
export_for_tableau(Xtrain, model, export_cols, "traffic_data.csv")
files.download("traffic_data.csv")

Saved 61839 rows to traffic_data.csv
Columns: ['geohash', 'day', 'time_minutes', 'hour', 'time_label', 'predicted_demand', 'congestion_level', 'Temperature', 'NumberofLanes', 'LargeVehicles', 'Landmarks']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Predicting the Testing value

In [212]:
dft.head()

,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy


In [213]:
dft.columns

Index(['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes',
       'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'],
      dtype='object')

In [214]:
dftn = dft.copy()

dftn = gp.fit_transform(dftn)
dftn = time_change(dftn)
dftn = labelEncode(dftn)
dftn, _, _, _, _ = preprocess(dftn)

In [215]:
dftn.head()

,Index,geohash,day,NumberofLanes,LargeVehicles,Landmarks,Temperature,lat,lon,geo_zone,...,night_time,RoadType_Highway,RoadType_Residential,RoadType_Street,RoadType_nan,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_nan
0,0,qp02z1,49,1,1,0,16.457339,-5.484924,90.664673,qp02z,...,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1,qp02z9,49,1,1,0,6.476213,-5.484924,90.686646,qp02z,...,0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,2,qp02yf,49,3,0,1,22.318203,-5.479431,90.653687,qp02y,...,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,3,qp02z6,49,2,1,1,16.457339,-5.479431,90.675659,qp02z,...,0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,4,qp02zd,49,1,1,0,18.266162,-5.479431,90.686646,qp02z,...,0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [228]:
dftn["geohash"] = le_geohash.transform(dftn["geohash"])

In [229]:
le_geohash.classes_[:10]

array(['qp02yc', 'qp02yf', 'qp02yy', 'qp02yz', 'qp02z1', 'qp02z3',
       'qp02z4', 'qp02z5', 'qp02z6', 'qp02z7'], dtype=object)

In [230]:
dftn["geohash"].nunique()

1190

In [231]:
SLOTS_PER_DAY = 96

def add_lag_features(df, has_target=True,
                      lag_slots=(1, 4, 96),
                      roll_windows=(4, 96)):
    df = df.copy()
    df['_time_key'] = df['day'] * 1440 + df['time_minutes']
    df = df.sort_values(['geohash', '_time_key']).reset_index(drop=True)

    if not has_target:
        raise ValueError("add_lag_features needs a 'demand' column present.")

    grp = df.groupby('geohash')['demand']

    for lag in lag_slots:
      if lag == 1:
        df[f'demand_lag_15'] = grp.shift(lag)
      elif lag == 4:
        df[f'demand_lag_hour'] = grp.shift(lag)
      elif lag == 96:
        df[f'demand_lag_day'] = grp.shift(lag)

    for window in roll_windows:
        shifted = grp.shift(1)
        if window == 4:
          df[f'demand_roll_mean_hour'] = (
              shifted.groupby(df['geohash']).rolling(window).mean().reset_index(level=0, drop=True)
          )
          df[f'demand_roll_std_hour'] = (
              shifted.groupby(df['geohash']).rolling(window).std().reset_index(level=0, drop=True)
          )

    df = df.drop(columns=['_time_key'])
    return df

In [232]:
dfn["geohash"].nunique()

1249

In [233]:
dfcopy = dfn.copy()
dftcopy = dftn.copy()

In [234]:
dftcopy.columns

Index(['Index', 'geohash', 'day', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'lat', 'lon', 'geo_zone', 'dist_to_center',
       'geo_cluster', 'time_minutes', 'hour', 'sin_hour', 'cos_hour',
       'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'RoadType_Highway', 'RoadType_Residential', 'RoadType_Street',
       'RoadType_nan', 'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy',
       'Weather_Sunny', 'Weather_nan'],
      dtype='object')

In [235]:
dfcopy.columns

Index(['Index', 'geohash', 'day', 'demand', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan', 'display_lat', 'display_lon'],
      dtype='object')

In [236]:
dfcopy.head()

,Index,geohash,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,time_minutes,hour,...,RoadType_Residential,RoadType_Street,RoadType_nan,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_nan,display_lat,display_lon
36776,36776,0,48,0.046790,1,1,0,22.809028,630,10.50,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.846786,77.423714
37683,37683,0,48,0.021158,1,1,0,16.543659,645,10.75,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.846786,77.423714
2911,2911,0,48,0.005397,3,0,1,11.299877,60,1.00,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,12.846786,77.423714
8010,8010,0,48,0.012944,2,1,1,32.142120,150,2.50,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,12.846786,77.423714
8905,8905,0,48,0.025961,1,1,0,15.811063,165,2.75,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.846786,77.423714


In [237]:
dftcopy.head()

,Index,geohash,day,NumberofLanes,LargeVehicles,Landmarks,Temperature,lat,lon,geo_zone,...,night_time,RoadType_Highway,RoadType_Residential,RoadType_Street,RoadType_nan,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_nan
0,0,4,49,1,1,0,16.457339,-5.484924,90.664673,qp02z,...,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1,10,49,1,1,0,6.476213,-5.484924,90.686646,qp02z,...,0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,2,1,49,3,0,1,22.318203,-5.479431,90.653687,qp02y,...,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,3,8,49,2,1,1,16.457339,-5.479431,90.675659,qp02z,...,0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,4,12,49,1,1,0,18.266162,-5.479431,90.686646,qp02z,...,0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [238]:
dftcopy["geohash"].nunique()

1190

In [239]:
dfcopy = dfcopy.drop(["display_lat", "display_lon"], axis = 1)

In [240]:
dfcopy["is_test"] = False
dftcopy["is_test"] = True

In [241]:
combined= pd.concat([dfcopy, dftcopy.assign(demand=np.nan)], ignore_index=True)
combined = combined.sort_values(["geohash", "day", "time_minutes"])
combined = add_lag_features(combined, has_target = True)

In [242]:
# combined["is_test"] = False
# combined.loc[combined["Index"].isin(dftcopy["Index"]), "is_test"] = True

dftrain = combined[combined["is_test"] == False].drop(columns=["is_test"])
dftest= combined[combined["is_test"] == True].drop(columns=["is_test"])

In [243]:
print(dftest["geohash"].nunique())
print(dftrain["geohash"].nunique())

1190
1249


In [244]:
key = ["geohash", "time_minutes", "day"]
print("Duplicate in train:", dftrain.duplicated(subset=key).sum())
print("Duplicate in train:", dftest.duplicated(subset=key).sum())

Duplicate in train: 0
Duplicate in train: 0


In [245]:
len(dftrain.columns)

35

In [246]:
Xtest.columns

Index(['Index', 'geohash', 'day', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan'],
      dtype='object')

In [247]:
lst = []
lstx = list(Xtest.columns)
lstd = list(dftest.columns)

In [248]:
for i in lstd:
  if i not in lstx:
    lst.append(i)

print(lst)

['demand']


In [249]:
dftest.columns

Index(['Index', 'geohash', 'day', 'demand', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan'],
      dtype='object')

In [250]:
dftest = dftest.drop("demand", axis = 1)

In [251]:
dftrain["geohash"].nunique()

1249

In [252]:
dftest["geohash"].nunique()

1190

In [253]:
def recursive_predict(train_df, test_df, model, feature_cols,
                       lag_slots=(1, 4, 96), roll_windows=(4, 96)):
    train_df = train_df.copy()
    test_df = test_df.copy()

    max_lag = max(max(lag_slots), max(roll_windows))

    train_df['_time_key'] = train_df['day'] * 1440 + train_df['time_minutes']
    test_df['_time_key'] = test_df['day'] * 1440 + test_df['time_minutes']

    history = defaultdict(dict)

    train_sorted = train_df.sort_values(['geohash', '_time_key'])
    for gh, group in train_sorted.groupby('geohash'):
        tail = group.tail(max_lag)
        for _, row in tail.iterrows():
            history[gh][row['_time_key']] = row['demand']

    test_sorted = test_df.sort_values(['geohash', '_time_key']).reset_index(drop=True)
    predictions = []

    for idx, row in test_sorted.iterrows():
        gh = row['geohash']
        tkey = row['_time_key']
        hist = history[gh]

        for lag in lag_slots:
          if lag == 1:
            row['demand_lag_15'] = hist.get(tkey - lag, np.nan)
          elif lag == 4:
            row['demand_lag_hour'] = hist.get(tkey - lag, np.nan)
          elif lag == 96:
            row['demand_lag_day'] = hist.get(tkey - lag, np.nan)

        vals = [hist.get(tkey - 1 - 4, np.nan)]
        vals = [v for v in vals if not pd.isna(v)]
        row[f'demand_roll_mean_hour'] = np.mean(vals) if vals else np.nan
        row[f'demand_roll_std_hour'] = np.std(vals) if len(vals) > 1 else np.nan

        X_row = pd.DataFrame([row])[feature_cols]

        for col in ['geo_zone', 'geo_cluster']:
            if col in X_row.columns:
                X_row[col] = X_row[col].astype(pd.CategoricalDtype(categories=dftrain[col].cat.categories))

        pred = model.predict(X_row)[0]

        pred = model.predict(X_row)[0]
        predictions.append(pred)

        history[gh][tkey] = pred
        test_sorted.loc[idx, feature_cols] = row[feature_cols].values

    test_sorted['predicted_demand'] = predictions
    return test_sorted.drop(columns=['_time_key'])

In [254]:
def recursive_predict_fast(train_df, test_df, model, feature_cols,
                            lag_slots=(15, 60, 1440),      # in MINUTES now, not slots
                            lag_names=('demand_lag_15', 'demand_lag_hour', 'demand_lag_day'),
                            roll_window_minutes=60,
                            roll_mean_name='demand_roll_mean_hour',
                            roll_std_name='demand_roll_std_hour',
                            time_step_minutes=15):
    train_df = train_df.copy()
    test_df = test_df.copy()

    train_df['_time_key'] = train_df['day'] * 1440 + train_df['time_minutes']
    test_df['_time_key'] = test_df['day'] * 1440 + test_df['time_minutes']

    # history[geohash][time_key] = demand   (seeded with real train values)
    history = defaultdict(dict)
    max_lookback = max(max(lag_slots), roll_window_minutes)
    train_sorted = train_df.sort_values(['geohash', '_time_key'])
    for gh, group in train_sorted.groupby('geohash'):
        recent = group[group['_time_key'] >= group['_time_key'].max() - max_lookback - 10]
        for _, row in recent.iterrows():
            history[gh][row['_time_key']] = row['demand']

    # process test set ONE TIME STEP AT A TIME, across all geohashes at once
    time_keys_sorted = sorted(test_df['_time_key'].unique())
    result_frames = []

    for tkey in time_keys_sorted:
        batch = test_df[test_df['_time_key'] == tkey].copy()

        # compute lag features for this whole batch using the history dict
        for lag_min, lag_name in zip(lag_slots, lag_names):
            batch[lag_name] = batch['geohash'].map(
                lambda gh: history[gh].get(tkey - lag_min, np.nan)
            )

        n_roll_steps = roll_window_minutes // time_step_minutes
        def roll_stats(gh):
            vals = [history[gh].get(tkey - time_step_minutes * (i + 1), np.nan)
                    for i in range(n_roll_steps)]
            vals = [v for v in vals if not pd.isna(v)]
            mean = np.mean(vals) if vals else np.nan
            std = np.std(vals) if len(vals) > 1 else np.nan
            return pd.Series({roll_mean_name: mean, roll_std_name: std})

        roll_df = batch['geohash'].apply(roll_stats)
        batch[roll_mean_name] = roll_df[roll_mean_name].values
        batch[roll_std_name] = roll_df[roll_std_name].values

        # batch predict ALL geohashes at this timestamp in ONE call
        X_batch = batch[feature_cols]
        preds = model.predict(X_batch)
        batch['predicted_demand'] = preds

        # push predictions into history so LATER timestamps can use them
        for gh, pred in zip(batch['geohash'], preds):
            history[gh][tkey] = pred

        result_frames.append(batch)

    result = pd.concat(result_frames, ignore_index=True)
    return result.drop(columns=['_time_key'])

In [255]:
col_features = ['Index', 'geohash', 'day', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan']

In [256]:
len(col_features)

34

In [257]:
N_timestamp = dftest["time_minutes"].nunique()*dftest["day"].nunique()
print(N_timestamp)

47


In [258]:
import time
df_sample = dftest.head(200)
t0 = time.time()
result = recursive_predict_fast(dftrain, df_sample, model, col_features)
elapsed = time.time() - t0

print(f"The time for {elapsed:.2f}s for 200 rows, estimated time ->  {elapsed/200*(dftest.size)/60:.1f}min")

The time for 6.05s for 200 rows, estimated time ->  716.1min


In [259]:
dftst = recursive_predict_fast(dftrain, dftest, model, col_features)

In [260]:
dftst.head()

,Index,geohash,day,NumberofLanes,LargeVehicles,Landmarks,Temperature,time_minutes,hour,sin_hour,...,RoadType_Highway,RoadType_Residential,RoadType_Street,RoadType_nan,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_nan,predicted_demand
0,2,1,49,3,0,1,22.318203,135,2.25,0.55557,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.025565
1,0,4,49,1,1,0,16.457339,135,2.25,0.55557,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.060304
2,3,8,49,2,1,1,16.457339,135,2.25,0.55557,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.036220
3,1,10,49,1,1,0,6.476213,135,2.25,0.55557,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.031786
4,4,12,49,1,1,0,18.266162,135,2.25,0.55557,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.059071


In [261]:
dftst.columns

Index(['Index', 'geohash', 'day', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'time_minutes', 'hour', 'sin_hour',
       'cos_hour', 'sin_minute', 'cos_minute', 'is_rush_hour', 'night_time',
       'lat', 'lon', 'geo_zone', 'dist_to_center', 'geo_cluster',
       'demand_lag_15', 'demand_lag_hour', 'demand_lag_day',
       'demand_roll_mean_hour', 'demand_roll_std_hour', 'RoadType_Highway',
       'RoadType_Residential', 'RoadType_Street', 'RoadType_nan',
       'Weather_Foggy', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny',
       'Weather_nan', 'predicted_demand'],
      dtype='object')

In [263]:
dft.columns

Index(['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes',
       'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'],
      dtype='object')

In [264]:
print(dft["geohash"].nunique())
print(dftst["geohash"].nunique())

1190
1190


In [265]:
dftst["geohash"] = le_geohash.inverse_transform(dftst["geohash"])

In [266]:
print(le_geohash.classes_.dtype)

object


In [267]:
dft["time_minutes"] = pd.to_datetime(dft["timestamp"], format="%H:%M").dt.hour*60 + pd.to_datetime(dft["timestamp"], format="%H:%M").dt.minute

In [268]:
dftst["geohash"].head()

,geohash
0,qp02yf
1,qp02z1
2,qp02z6
3,qp02z9
4,qp02zd


In [269]:
key = ["geohash", "day", "time_minutes"]
final = dft.merge(
    dftst[
          [
            "geohash",
            "day",
            "time_minutes",
            "predicted_demand"
          ]
        ], on=key, how="left")

In [287]:
dfinal.tail()

,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,time_minutes,predicted_demand,lat,lon,geo_zone,dist_to_center,geo_cluster
41773,41773,qp0d4q,49,13:45,Street,1,Not Allowed,Yes,19.588991,Sunny,825,0.253105,-5.237732,90.807495,qp0d4,0.119173,12
41774,41774,qp0d4w,49,13:45,Residential,2,Not Allowed,Yes,10.735538,Rainy,825,0.195928,-5.237732,90.818481,qp0d4,0.123220,12
41775,41775,qp0dhq,49,13:45,Residential,2,Not Allowed,Yes,13.223750,Rainy,825,0.015014,-5.237732,90.895386,qp0dh,0.169733,9
41776,41776,qp0dhw,49,13:45,Residential,2,Not Allowed,Yes,12.510917,Rainy,825,0.141913,-5.237732,90.906372,qp0dh,0.178105,9
41777,41777,qp0djq,49,13:45,Residential,2,Not Allowed,Yes,20.363693,Sunny,825,0.014730,-5.237732,90.939331,qp0dj,0.204710,8


In [300]:
lst = list(dfinal["predicted_demand"].unique())
nlst = []
for i in lst:
  if i > 0.5:
    nlst.append(i);
len(nlst)

1012

In [285]:
dfinal = gp.fit_transform(final)

In [292]:
dftraffic = rescale_to_city_bbox(dfinal)

In [294]:
dftraffic.shape

(41778, 19)

#checking the size

In [295]:
dftraffic.to_csv("trafficd.csv", index = False)

In [296]:
files.download("trafficd.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(len(dftst))
print(dftst["Index"].nunique())

83556
41778


In [ ]:
print(dftst["Index"].duplicated().sum())

41778


In [ ]:
dftst[dftst["Index"].duplicated(keep=False)].sort_values("Index").head(20)

,Index,geohash,day,NumberofLanes,LargeVehicles,Landmarks,Temperature,time_minutes,hour,sin_hour,...,RoadType_Highway,RoadType_Residential,RoadType_Street,RoadType_nan,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_nan,predicted_demand
0,0,4,48,1,1,0,16.405354,0,0.00,0.00000,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.071534
41779,0,3,49,1,1,0,16.457339,135,2.25,0.55557,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.061152
41781,1,9,49,1,1,0,6.476213,135,2.25,0.55557,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.032997
4,1,25,48,3,0,1,31.104565,0,0.00,0.00000,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.046918
235,2,370,48,1,1,0,25.919267,0,0.00,0.00000,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.040914
41778,2,1,49,3,0,1,22.318203,135,2.25,0.55557,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.025565
245,3,416,48,1,1,0,16.405354,0,0.00,0.00000,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.028824
41780,3,7,49,2,1,1,16.457339,135,2.25,0.55557,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.035952
2,4,22,48,1,1,0,10.803667,0,0.00,0.00000,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.046916
41782,4,11,49,1,1,0,18.266162,135,2.25,0.55557,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.035850


In [ ]:
key = ["geohash", "day", "time_minutes"]
print(dftst.duplicated(subset=key).sum())

0


In [ ]:
combined.size

In [ ]:
dft.columns

Index(['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes',
       'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'],
      dtype='object')

In [ ]:
dftn.head()

,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,cos_hour,sin_minute,cos_minute,is_rush_hour,night_time,lat,lon,geo_zone,dist_to_center,geo_cluster
0,0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN,...,0.83147,0.55557,0.83147,0,0,-5.484924,90.664673,qp02z,0.169923,0
1,1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy,...,0.83147,0.55557,0.83147,0,0,-5.484924,90.686646,qp02z,0.157483,0
2,2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny,...,0.83147,0.55557,0.83147,0,0,-5.479431,90.653687,qp02y,0.172695,0
3,3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy,...,0.83147,0.55557,0.83147,0,0,-5.479431,90.675659,qp02z,0.158958,0
4,4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy,...,0.83147,0.55557,0.83147,0,0,-5.479431,90.686646,qp02z,0.152813,0


In [ ]:
print("train_day", df["day"].value_counts())
print("test_day", dft["day"].value_counts())

print("train_time", df["timestamp"].value_counts())
print("test_time", dft["timestamp"].value_counts())

train_day day
48    69427
49     7872
Name: count, dtype: int64
test_day day
49    41778
Name: count, dtype: int64
train_time timestamp
2:0      1778
1:45     1755
1:30     1750
1:15     1698
1:0      1668
         ... 
19:30     308
18:30     303
18:45     297
19:0      292
19:15     288
Name: count, Length: 96, dtype: int64
test_time timestamp
4:30     939
4:45     932
5:0      932
4:0      928
4:15     926
5:30     926
5:15     923
9:0      922
5:45     918
2:30     915
3:45     909
10:0     909
3:30     908
6:15     904
9:15     904
9:30     904
7:15     903
6:0      901
8:45     901
9:45     898
6:45     897
11:0     897
3:0      897
2:45     896
3:15     896
8:0      895
6:30     894
7:0      893
7:30     892
10:30    890
7:45     889
10:15    889
2:15     887
8:30     884
8:15     881
10:45    880
12:0     878
11:30    876
11:45    871
11:15    870
12:15    856
12:30    847
12:45    834
13:0     815
13:15    805
13:30    790
13:45    777
Name: count, dtype: int64


In [ ]:
dft = dft.drop("timestamp", axis = 1)
dftime = dft["timestamp_minute"]
dft = dft.drop("timestamp_minute", axis = 1)
dft = pd.concat([dft, dftime], axis = 1)

In [ ]:
Y_pred = model.predict(dftest)
print(Y_pred)

In [ ]:
submission = pd.DataFrame({"Index": dft["Index"], "demand":Y_pred})